# Modelo principal: DINOv2-L + Ridge en la partición train/val/test

En esta libreta entrenamos y evaluamos el modelo que aprobamos en Evaluation: DINOv2-L congelado a 448 px (token CLS más el promedio de los parches) con una regresión Ridge encima, usando los recortes de SAM con CLAHE.

Pasos:
1. Configuración y carga de los recortes del rancho.
2. Partición única por animal en train, val y test (`grouped_holdout`, semilla 42, la misma función de los notebooks de YOLO).
3. Extracción de características con DINOv2-L congelado (se guarda en caché).
4. Entrenamiento de Ridge con train y elección de `alpha` con val.
5. Evaluación en test, una sola vez.
6. Guardado del modelo y de las predicciones (fuera del repositorio).

Las fotos y las etiquetas son del CAETEC y son confidenciales, así que no están en el repositorio. Para correr la libreta hace falta la carpeta de recortes que genera `SAM.ipynb` y la tabla `imagenes.csv` del Drive del equipo. Las salidas que se muestran aquí son solo agregados, sin identificadores de vacas.

## 1. Configuración

In [1]:
from pathlib import Path
import numpy as np

# Rutas: carpeta con recortes/recortes.csv (salida de SAM.ipynb) y la tabla de imágenes
EN_COLAB = False
if EN_COLAB:
    DATOS = Path('/content/datos')
    RUTA_TABLA = Path('/content/drive/MyDrive/lecheros/Dataset/imagenes.csv')
else:
    DATOS = Path.home() / 'bcs_data' / 'nube' / 'datos'
    RUTA_TABLA = Path.home() / 'bcs_data' / 'imagenes.csv'
SALIDA = DATOS.parent / 'modelo_principal'      # caché, modelo y predicciones; fuera del repositorio

SEMILLA = 42
FRACCION_TEST, FRACCION_VAL = 0.20, 0.20        # val es el 20 % de lo que queda después de test
EXTRACTOR, TAMANO = 'vit_large_patch14_dinov2.lvd142m', 448
VARIANTE = 'archivo_mascara_clahe'              # recorte con máscara de SAM + CLAHE
ALPHAS = np.logspace(-1, 5, 25)

# Criterios de éxito (Business Understanding)
OBJ_MAE, OBJ_RMSE, OBJ_TOLERANCIA = 0.25, 0.35, 80.0
INTERVALO_ORIGINAL = (3.25, 3.50)
PASO, ESCALA = 0.25, (1.0, 5.0)

In [2]:
import re, time, warnings
import joblib, pandas as pd, timm, torch
from IPython.display import display
from PIL import Image
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
SALIDA.mkdir(parents=True, exist_ok=True)
print('Dispositivo:', DISPOSITIVO)

Dispositivo: mps


## 2. Datos

Aplicamos los mismos filtros que en `Modelo_CNN_Vacuno.ipynb` y `Comparacion_Modelos_BCS.ipynb`: recortes `ok` del rancho, sin vistas laterales y con `usar = TRUE` en la tabla. Una sesión es una vaca en una visita (`id_vaca`) y un animal es el arete. Cada sesión pesa 1 en el entrenamiento, repartido entre sus fotos, para que las vacas con muchas fotos no dominen.

In [3]:
def a_bool(v):
    return str(v).strip().lower() in {'true', '1', 'verdadero', 'si', 'sí', 'yes', 'x'}

def a_relativa(r):
    m = re.search(r'(?:^|/)(clean/.+)$', str(r).replace('\\', '/'))
    return m.group(1) if m else None

def limpiar_id(s):
    return s.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

rec = pd.read_csv(DATOS / 'recortes' / 'recortes.csv', dtype={'id_vaca': str, 'animal': str})
tab = pd.read_csv(RUTA_TABLA)
tab['archivo_limpio'] = tab['archivo_limpio'].map(a_relativa)
tab = tab.dropna(subset=['archivo_limpio']).drop_duplicates('archivo_limpio')
tab['animal'], tab['usar_tabla'] = limpiar_id(tab['animal']), tab['usar'].map(a_bool)
rec = rec.drop(columns=['animal', 'ronda'], errors='ignore').merge(
    tab[['archivo_limpio', 'animal', 'ronda', 'usar_tabla']], on='archivo_limpio', how='left')
rec['id_vaca'] = limpiar_id(rec['id_vaca'])
datos = rec[(rec['estado'] == 'ok') & (rec['fuente'] == 'rancho') & (rec['vista'].fillna('') != 'lateral')
            & (rec['usar_tabla'] == True)].reset_index(drop=True).copy()
datos['bcs'] = datos['bcs'].astype(float)
Y = datos['bcs'].to_numpy()
PESO = (1 / datos.groupby('id_vaca')['bcs'].transform('size')).to_numpy()
print(f"{len(datos)} fotos, {datos['animal'].nunique()} animales, {datos['id_vaca'].nunique()} sesiones")

604 fotos, 80 animales, 93 sesiones


## 3. Partición train/val/test por animal

`grouped_holdout` prueba 2000 particiones aleatorias por animal y se queda con la que mejor conserva la distribución de BCS del total (y la fracción pedida). Primero separamos el 20 % para test y luego, de lo que queda, el 20 % para val. Todas las fotos de un animal quedan en la misma partición.

In [4]:
def grouped_holdout(frame, fraction, seed):
    frame = frame.reset_index(drop=True)
    clases = sorted(frame['bcs'].unique())
    total = frame['bcs'].value_counts(normalize=True).reindex(clases, fill_value=0)
    mejor = None
    for fit_idx, held_idx in GroupShuffleSplit(n_splits=2000, test_size=fraction, random_state=seed).split(frame, groups=frame['animal']):
        fit, held = frame.iloc[fit_idx], frame.iloc[held_idx]
        if not set(clases).issubset(set(fit['bcs'])):
            continue
        dist = held['bcs'].value_counts(normalize=True).reindex(clases, fill_value=0)
        costo = float(np.abs(dist - total).sum() + abs(len(held) / len(frame) - fraction))
        if mejor is None or costo < mejor[0]:
            mejor = (costo, fit.copy(), held.copy())
    return mejor[1], mejor[2]

desarrollo, test = grouped_holdout(datos, FRACCION_TEST, SEMILLA)
fit, val = grouped_holdout(desarrollo, FRACCION_VAL, SEMILLA)
asignacion = {a: s for s, f in [('train', fit), ('val', val), ('test', test)] for a in f['animal'].unique()}
datos['split'] = datos['animal'].map(asignacion)

assert datos['split'].notna().all()
assert datos.groupby('animal')['split'].nunique().max() == 1, 'un animal quedó en dos particiones'
assert set(datos['bcs']) <= set(datos.loc[datos['split'] == 'train', 'bcs']), 'falta una clase en train'

tr, va, te = (np.where(datos['split'] == s)[0] for s in ['train', 'val', 'test'])
display(datos.groupby('split').agg(fotos=('bcs', 'size'), animales=('animal', 'nunique'), sesiones=('id_vaca', 'nunique'),
                                   bcs_medio=('bcs', 'mean'), bcs_min=('bcs', 'min'), bcs_max=('bcs', 'max')).round(2))
display(pd.crosstab(datos['bcs'], datos['split']))

,fotos,animales,sesiones,bcs_medio,bcs_min,bcs_max
split,,,,,,
test,100,16,19,3.04,2.50,3.75
train,402,51,59,3.00,2.00,3.75
val,102,13,15,3.05,2.25,3.75


split,test,train,val
bcs,,,
2.00,0,4,0
2.25,0,2,4
2.50,13,70,14
2.75,28,96,18
3.00,20,85,24
3.25,17,67,16
3.50,11,38,16
3.75,11,40,10


## 4. Características de DINOv2-L congelado

Cada recorte se redimensiona a 448 × 448, se normaliza con las medias de ImageNet y pasa por DINOv2-L sin entrenarlo. Tomamos el token `CLS` y el promedio de los parches (2 × 1024 = 2048 valores) y promediamos la foto con su versión en espejo. No hay augmentation: el extractor no se entrena, solo la regresión de encima.

La primera vez tarda unos minutos (en una GPU o en MPS); después se lee del archivo de caché.

In [5]:
CACHE = SALIDA / f'dinov2_l_{TAMANO}_{VARIANTE}_cls_parches.npz'
MEDIA = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
DESV = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

@torch.no_grad()
def caracteristicas(modelo, x):
    t = modelo.forward_features(x)
    cls, parches = t[:, 0], t[:, 1 + getattr(modelo, 'num_reg_tokens', 0):].mean(1)
    return torch.cat([cls, parches], 1)

if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    assert (z['archivo'] == datos['archivo_limpio'].to_numpy()).all(), 'la caché no corresponde a estos datos'
    X = z['X']
    print('Características leídas de la caché:', X.shape)
else:
    modelo = timm.create_model(EXTRACTOR, pretrained=True, num_classes=0, img_size=TAMANO).eval().to(DISPOSITIVO)
    lotes, t0 = [], time.time()
    for i in range(0, len(datos), 16):
        imgs = [np.asarray(Image.open(DATOS / r).convert('RGB').resize((TAMANO, TAMANO), Image.BICUBIC))
                for r in datos[VARIANTE].iloc[i:i + 16]]
        x = ((torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float() / 255 - MEDIA) / DESV).to(DISPOSITIVO)
        lotes.append(((caracteristicas(modelo, x) + caracteristicas(modelo, torch.flip(x, dims=[3]))) / 2).float().cpu().numpy())
    X = np.concatenate(lotes)
    np.savez(CACHE, X=X, archivo=datos['archivo_limpio'].to_numpy())
    del modelo
    print(f'Características extraídas: {X.shape} en {time.time() - t0:.0f} s')

Características extraídas: (604, 2048) en 245 s


## 5. Ridge: entrenamiento con train y elección de alpha con val

Estandarizamos las características y ajustamos Ridge con los pesos por sesión. Probamos 25 valores de `alpha` entre 0.1 y 100 000 y elegimos el que da menor MAE ponderado en val. Test no se usa en ningún momento de esta elección.

In [6]:
def mae_ponderado(y, p, w):
    return np.sum(w * np.abs(y - p)) / np.sum(w)

def ajustar(idx, alpha):
    m = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    m.fit(X[idx], Y[idx], ridge__sample_weight=PESO[idx])
    return m

errores = [mae_ponderado(Y[va], ajustar(tr, a).predict(X[va]), PESO[va]) for a in ALPHAS]
ALPHA = ALPHAS[int(np.argmin(errores))]
modelo_final = ajustar(tr, ALPHA)
pred = modelo_final.predict(X)
print(f'alpha elegido: {ALPHA:.1f}  (MAE ponderado en val {min(errores):.3f})')

alpha elegido: 177.8  (MAE ponderado en val 0.263)


## 6. Evaluación en test

Métricas por foto (como pide el criterio) y por vaca (promedio de las fotos de cada sesión). El acierto ±0.25 usa la predicción redondeada al paso de 0.25. Los números deben coincidir con los del reporte de Evaluation: MAE 0.210, RMSE 0.265 y 81 % por foto.

In [7]:
redondear = lambda p: np.clip(np.round(np.asarray(p) / PASO) * PASO, *ESCALA)

def metricas(y, p):
    return {'n': len(y), 'MAE': np.abs(p - y).mean(), 'RMSE': np.sqrt(((p - y) ** 2).mean()),
            '±0.25 %': 100 * np.mean(np.abs(redondear(p) - y) <= PASO + 1e-9)}

res = datos[['id_vaca', 'bcs', 'split']].assign(pred=pred)
filas = []
for particion in ['train', 'val', 'test']:
    foto = res[res['split'] == particion]
    vaca = foto.groupby('id_vaca').agg(bcs=('bcs', 'first'), pred=('pred', 'mean'))
    for nivel, df in [('foto', foto), ('vaca', vaca)]:
        for criterio, sub in [('rango completo', df), ('original 3.25–3.50', df[df['bcs'].between(*INTERVALO_ORIGINAL)])]:
            m = metricas(sub['bcs'].to_numpy(), sub['pred'].to_numpy())
            m['cumple'] = m['MAE'] <= OBJ_MAE and m['RMSE'] <= OBJ_RMSE and m['±0.25 %'] >= OBJ_TOLERANCIA
            filas.append({'partición': particion, 'nivel': nivel, 'criterio': criterio, **m})
tabla = pd.DataFrame(filas)
display(tabla[tabla['partición'] == 'test'].round(3))
display(tabla[(tabla['criterio'] == 'rango completo') & (tabla['nivel'] == 'foto')].round(3))

,partición,nivel,criterio,n,MAE,RMSE,±0.25 %,cumple
8,test,foto,rango completo,100,0.210,0.265,81.000,True
9,test,foto,original 3.25–3.50,28,0.348,0.374,57.143,False
10,test,vaca,rango completo,19,0.186,0.231,89.474,True
11,test,vaca,original 3.25–3.50,5,0.320,0.327,80.000,False


,partición,nivel,criterio,n,MAE,RMSE,±0.25 %,cumple
0,train,foto,rango completo,402,0.131,0.172,95.522,True
4,val,foto,rango completo,102,0.228,0.273,83.333,True
8,test,foto,rango completo,100,0.210,0.265,81.000,True


## 7. Guardado del modelo y de las predicciones

El modelo entrenado (escalador + Ridge) y las predicciones se guardan en `SALIDA`, fuera del repositorio, porque las predicciones llevan los identificadores de las vacas. Para usar el modelo con una foto nueva hay que recortarla con SAM, aplicar CLAHE, extraer las características igual que en el paso 4 y llamar a `modelo.predict`.

In [8]:
joblib.dump({'modelo': modelo_final, 'alpha': ALPHA, 'extractor': EXTRACTOR, 'tamano': TAMANO,
             'variante': VARIANTE, 'representacion': 'cls + promedio de parches, foto y espejo promediados'},
            SALIDA / 'dinov2_l_ridge.joblib')
res.assign(archivo_limpio=datos['archivo_limpio'], animal=datos['animal']).to_csv(SALIDA / 'pred_dinov2_l_ridge.csv', index=False)
print('Guardado en la carpeta de salida:', sorted(p.name for p in SALIDA.iterdir()))

Guardado en la carpeta de salida: ['dinov2_l_448_archivo_mascara_clahe_cls_parches.npz', 'dinov2_l_ridge.joblib', 'pred_dinov2_l_ridge.csv']
